## 01. 교차검증

모델을 한 번만 train/test로 나누면 평가 점수가 데이터 분할에 따라 달라질 수 있다.
특히 데이터가 적거나 class 비율이 한쪽으로 치우친 경우에는 한 번의 점수만 보고 모델을 선택하기 어렵다.

**배우는 이유**
- 모델 성능을 더 안정적으로 추정하기 위함.
- 하이퍼파라미터 튜닝에서 테스트 데이터를 반복 사용하지 않기 위함.
- 여러 모델을 공정하게 비교하기 위함.

**어디서 사용하는가?**
- GridSearchCV, RandomizedSearchCV 같은 튜닝 도구 내부.
- 모델 후보를 비교할 때.
- 데이터가 많지 않아 검증셋을 따로 크게 떼기 어려울 때.

**핵심 용어**
- fold: 교차검증에서 데이터를 나눈 하나의 묶음.
- K-Fold: 데이터를 K개 fold로 나누고, 각 fold를 한 번씩 검증용으로 사용하는 방법.
- Stratified K-Fold: 분류 문제에서 class 비율을 유지하며 fold를 나누는 방법.
- validation score: 학습 데이터 내부 검증에서 얻은 점수.
- test score: 최종 평가 데이터에서 마지막으로 확인하는 점수.

## 02. 실습 환경 준비

Iris 분류 데이터와 Diabetes 회귀 데이터를 사용한다.
교차검증 함수, Pipeline, 스케일링, 분류/회귀 모델을 함께 사용해 검증 흐름을 확인한다.

In [1]:
import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import single

from sklearn.datasets import load_iris, load_diabetes

from sklearn.model_selection import train_test_split, KFold, StratifiedKFold, cross_val_score, cross_validate

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression, Ridge

from sklearn.metrics import accuracy_score, f1_score

## 03. Iris 데이터 로드

Iris 데이터는 꽃받침/꽃잎 길이와 너비를 이용해 3개 품종 중 하나를 예측하는 분류 데이터셋이다.
교차검증에서는 class 비율이 fold마다 유지되는지 확인하기 좋다.

In [2]:
iris = load_iris(as_frame=True)

iris_X = iris.data
iris_y = iris.target

print('feature shape:', iris_X.shape)
print('target shape:', iris_y.shape)
print('target names:', iris.target_names)
display(iris_X.head())
display(iris_y.value_counts().sort_index())


feature shape: (150, 4)
target shape: (150,)
target names: ['setosa' 'versicolor' 'virginica']


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2


target
0    50
1    50
2    50
Name: count, dtype: int64

## 04. 학습/평가 데이터 분리

교차검증은 학습 데이터 내부에서 수행하고, 테스트 데이터는 마지막 확인용으로 남겨둔다.
`stratify=iris_y`를 사용하면 train/test에도 class 비율이 유지된다.

In [3]:
iris_X_train, iris_X_test, iris_y_train, iris_y_test = train_test_split(
    iris_X,
    iris_y,
    test_size=0.2,
    random_state=42,
    stratify=iris_y
)

print('train:', iris_X_train.shape, iris_y_train.shape)
print('test:', iris_X_test.shape, iris_y_test.shape)
print('train class count')
display(iris_y_train.value_counts().sort_index()) # 40 / 40 / 40
print('test class count')
display(iris_y_test.value_counts().sort_index()) # 10 / 10 / 10


train: (120, 4) (120,)
test: (30, 4) (30,)
train class count


target
0    40
1    40
2    40
Name: count, dtype: int64

test class count


target
0    10
1    10
2    10
Name: count, dtype: int64

## 05. KFold와 StratifiedKFold 분할 비교

`KFold`는 단순히 데이터를 K개 묶음으로 나누고, `StratifiedKFold`는 class 비율을 유지하면서 나눈다.
분류 문제에서는 보통 `StratifiedKFold`를 우선 사용한다.

In [8]:

# KFold : target class의 비율을 고려하지 않고 fold를 나눔
plain_kfold = KFold(n_splits=5, random_state=42, shuffle=True)

# StratifiedKFold : 각 fold에 target class 비율이 최대한 비슷하게 나눔
stratified_kfold = StratifiedKFold(n_splits=5, random_state=42, shuffle=True)

folds = [('KFold', plain_kfold), ('StratifiedKFold', stratified_kfold)]

fold_rows = []

for method_name, cv in folds:

    # cv.split() : train index와 validation index를 fold별로 반환
    cv_split = cv.split(iris_X_train, iris_y_train)
    for fold_no, (_, valid_idx) in enumerate(cv_split, start=1):
        valid_target = iris_y_train.iloc[valid_idx]
        class_counts = valid_target.value_counts().sort_index()

        fold_rows.append({
            'method': method_name,
            'fold': fold_no,
            'class_0': class_counts.get(0, 0),
            'class_1': class_counts.get(1, 0),
            'class_2': class_counts.get(2, 0),
            'valid_size': len(valid_idx)
        })

        # print(f'{method_name} fold {fold_no}:')
        # print(f'valid_idx: {valid_idx}')
        # print(f'valid_target: {valid_target}')
        # print(f'class counts: {class_counts}')
fold_df = pd.DataFrame(fold_rows)
display(fold_df)

,method,fold,class_0,class_1,class_2,valid_size
0,KFold,1,8,9,7,24
1,KFold,2,7,8,9,24
2,KFold,3,8,5,11,24
3,KFold,4,8,8,8,24
4,KFold,5,9,10,5,24
5,StratifiedKFold,1,8,8,8,24
6,StratifiedKFold,2,8,8,8,24
7,StratifiedKFold,3,8,8,8,24
8,StratifiedKFold,4,8,8,8,24
9,StratifiedKFold,5,8,8,8,24


## 06. 단일 train/test 평가와 교차검증 비교

먼저 단일 train/test split으로 모델을 평가한 뒤, 같은 모델을 교차검증으로 평가한다.
두 결과를 비교하면 한 번의 점수보다 여러 fold 평균을 보는 이유가 드러남.

In [17]:
tree_clf = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

# 1. train 학습, test 평가
tree_clf.fit(iris_X_train, iris_y_train)


single_pred = tree_clf.predict(iris_X_test)

# 정확도 평가
# single_test_accuracy = tree_clf.score(iris_X_test, iris_y_test)
single_test_accuracy = accuracy_score(iris_y_test, single_pred)


# 2. cross_val_score()
# - 학습 데이타를 여러 fold로 나누어 검증 점수 배열을 반환
cv_scores = cross_val_score(
    tree_clf,
    iris_X_train,
    iris_y_train,
    cv=stratified_kfold,
    scoring='accuracy'
)


print("단일 test accuracy: ", round(single_test_accuracy, 4))
print("fold별 validation accuracy: ", np.round(cv_scores, 4))
print("교차검증 평균: ", round(cv_scores.mean(), 4))
print("교차검증 표준편차: ", round(cv_scores.std(), 4))

단일 test accuracy:  0.9667
fold별 validation accuracy:  [0.9583 1.     0.9583 0.9583 0.9167]
교차검증 평균:  0.9583
교차검증 표준편차:  0.0264


## 07. Pipeline과 cross_validate

`cross_validate()`는 여러 평가 지표와 학습/검증 시간을 함께 반환한다.
스케일링이 필요한 모델은 `Pipeline`으로 전처리와 모델을 묶어 교차검증 안에서 안전하게 처리한다.

In [18]:
logistic_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=3000))
])

# scoring: 여러 지표를 dict 형태로 전달하면 cross_validate가 지표별 결과를 반환한다.
scoring = {
    'accuracy': 'accuracy',
    'f1_macro': 'f1_macro'
}

# return_train_score=True: train fold 점수도 함께 반환해 과대적합 여부를 참고할 수 있다.
cv_result = cross_validate(
    logistic_pipeline,
    iris_X_train,
    iris_y_train,
    cv=stratified_kfold,
    scoring=scoring,
    return_train_score=True
)

cv_result_df = pd.DataFrame(cv_result)
display(cv_result_df.round(4))

summary_df = pd.DataFrame({
    'metric': ['train_accuracy', 'validation_accuracy', 'validation_f1_macro'],
    'mean': [
        cv_result_df['train_accuracy'].mean(),
        cv_result_df['test_accuracy'].mean(),
        cv_result_df['test_f1_macro'].mean()
    ],
    'std': [
        cv_result_df['train_accuracy'].std(),
        cv_result_df['test_accuracy'].std(),
        cv_result_df['test_f1_macro'].std()
    ]
})
display(summary_df.round(4))

# train과 test의 점수 차이가 크면 과대적합을 의심
# f1_marco는 class별 성능을 균형있게 보는 지표이다
# -> accuracy와 함께 확인하여 클래스 불균형 확인의 지표가 된다

,fit_time,score_time,test_accuracy,train_accuracy,test_f1_macro,train_f1_macro
0,0.0251,0.0060,0.9583,0.9583,0.9582,0.9583
1,0.0080,0.0040,1.0000,0.9583,1.0000,0.9583
2,0.0079,0.0050,0.9583,0.9688,0.9582,0.9687
3,0.0069,0.0030,0.9583,0.9688,0.9582,0.9687
4,0.0060,0.0041,0.9167,0.9792,0.9153,0.9791


,metric,mean,std
0,train_accuracy,0.9667,0.0087
1,validation_accuracy,0.9583,0.0295
2,validation_f1_macro,0.9580,0.0299


## 08. 교차검증 후 최종 test 평가

교차검증으로 모델 후보의 성능을 추정한 뒤, 선택한 모델을 전체 train 데이터로 다시 학습하고 test 데이터로 마지막 평가한다.

In [21]:
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier


candidate_models = {
    'logistic_regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=3000))
    ]),
    'knn': Pipeline([
        ('scaler', StandardScaler()),
        ('model', KNeighborsClassifier(n_neighbors=5))
    ]),
    'svc_rbf': Pipeline([
        ('scaler', StandardScaler()),
        ('model', SVC(kernel='rbf', C=1.0, gamma='scale'))
    ]),
    'decision_tree': DecisionTreeClassifier(max_depth=3, random_state=42),
    'random_forest': RandomForestClassifier(n_estimators=100, random_state=42)
}

model_compare_rows = []

for model_name, model in candidate_models.items():
    cv_result = cross_validate(
        model,
        iris_X_train,
        iris_y_train,
        cv=stratified_kfold,
        scoring=scoring,
        return_train_score=True
    )

    # fold별 결과의 평균과 표준편차를 저장한다.
    # 평균은 성능 수준, 표준편차는 fold에 따른 점수 흔들림을 보는 값이다.
    model_compare_rows.append({
        'model': model_name,
        'train_accuracy_mean': cv_result['train_accuracy'].mean(),
        'validation_accuracy_mean': cv_result['test_accuracy'].mean(),
        'validation_accuracy_std': cv_result['test_accuracy'].std(),
        'validation_f1_macro_mean': cv_result['test_f1_macro'].mean(),
        'fit_time_mean': cv_result['fit_time'].mean()
    })


model_compare_df = pd.DataFrame(model_compare_rows)

# validation_f1_macro_mean을 우선 기준으로 정렬한다.
# Iris는 class가 균형적이지만, 여러 class를 고르게 맞히는지 보기 위해 f1_macro를 함께 사용한다.
model_compare_df = model_compare_df.sort_values(
    ['validation_f1_macro_mean', 'validation_accuracy_mean'],
    ascending=False
).reset_index(drop=True)

display(model_compare_df.round(4))

# 가장 위에 있는 모델을 최종 후보로 선택한다.
best_model_name = model_compare_df.loc[0, 'model']
best_model = candidate_models[best_model_name]

print('교차검증 기준 선택 모델:', best_model_name)

,model,train_accuracy_mean,validation_accuracy_mean,validation_accuracy_std,validation_f1_macro_mean,fit_time_mean
0,svc_rbf,0.9771,0.9667,0.0167,0.9665,0.0046
1,logistic_regression,0.9667,0.9583,0.0264,0.9580,0.0099
2,knn,0.9688,0.9583,0.0264,0.9580,0.0040
3,decision_tree,0.9792,0.9583,0.0264,0.9580,0.0020
4,random_forest,1.0000,0.9500,0.0312,0.9494,0.1237


교차검증 기준 선택 모델: svc_rbf


## 09. 회귀 모델의 교차검증

교차검증은 분류뿐 아니라 회귀에도 사용한다.
회귀에서는 `R2`, `RMSE`, `MAE` 같은 지표를 사용하며, scikit-learn의 오차 지표는 큰 값이 좋다는 규칙 때문에 음수로 반환되는 경우가 있다.

In [23]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import Ridge
from sklearn.datasets import load_diabetes
from sklearn.metrics import r2_score, root_mean_squared_error

# diabetes_X: 나이, 혈압, 혈액 검사 수치 등 입력 feature 데이터이다.
# diabetes_y: 1년 뒤 당뇨병 진행 정도를 나타내는 연속형 target이다.
diabetes = load_diabetes(as_frame=True)
diabetes_X = diabetes.data
diabetes_y = diabetes.target

diabetes_X_train, diabetes_X_test, diabetes_y_train, diabetes_y_test = train_test_split(
    diabetes_X,
    diabetes_y,
    test_size=0.2,
    random_state=42
)

regression_candidates = {
    'ridge': Pipeline([
        ('scaler', StandardScaler()),
        ('model', Ridge(alpha=1.0))
    ]),
    'knn_regressor': Pipeline([
        ('scaler', StandardScaler()),
        ('model', KNeighborsRegressor(n_neighbors=5))
    ])
}

regression_scoring = {
    'R2': 'r2',
    'RMSE': 'neg_root_mean_squared_error'
}

regression_compare_rows = []

for model_name, model in regression_candidates.items():
    # cross_validate(): 현재 회귀 모델 후보를 같은 fold와 같은 지표로 검증한다.
    cv_result = cross_validate(
        model,
        diabetes_X_train,
        diabetes_y_train,
        cv=5,
        scoring=regression_scoring,
        return_train_score=True
    )

    train_rmse_scores = -1 * cv_result['train_RMSE']
    validation_rmse_scores = -1 * cv_result['test_RMSE']

    regression_compare_rows.append({
        'model': model_name,
        'train_R2_mean': cv_result['train_R2'].mean(),
        'validation_R2_mean': cv_result['test_R2'].mean(),
        'validation_R2_std': cv_result['test_R2'].std(),
        'train_RMSE_mean': train_rmse_scores.mean(),
        'validation_RMSE_mean': validation_rmse_scores.mean(),
        'validation_RMSE_std': validation_rmse_scores.std()
    })

    # validation_RMSE_mean이 낮은 모델을 우선 선택하고, 동률에 가까우면 validation_R2_mean이 높은 모델을 참고한다.
regression_compare_df = pd.DataFrame(regression_compare_rows).sort_values(
    ['validation_RMSE_mean', 'validation_R2_mean'],
    ascending=[True, False]
).reset_index(drop=True)

display(regression_compare_df.round(4))

best_regression_model_name = regression_compare_df.loc[0, 'model']
best_regression_model = regression_candidates[best_regression_model_name]
print('교차검증 기준 선택 회귀 모델:', best_regression_model_name)

best_regression_model.fit(diabetes_X_train, diabetes_y_train)

diabetes_test_pred = best_regression_model.predict(diabetes_X_test)
print('최종 test R2:', round(r2_score(diabetes_y_test, diabetes_test_pred), 4))
print('최종 test RMSE:', round(root_mean_squared_error(diabetes_y_test, diabetes_test_pred), 4))

,model,train_R2_mean,validation_R2_mean,validation_R2_std,train_RMSE_mean,validation_RMSE_mean,validation_RMSE_std
0,ridge,0.5306,0.4513,0.1388,53.3030,55.9165,2.9786
1,knn_regressor,0.5744,0.3175,0.1014,50.7859,62.9360,2.6387


교차검증 기준 선택 회귀 모델: ridge
최종 test R2: 0.4541
최종 test RMSE: 53.7775


## 10. 정리

- 교차검증은 학습 데이터를 여러 fold로 나누어 모델 성능을 더 안정적으로 추정하는 방법임.
- 분류 문제에서는 class 비율을 유지하는 StratifiedKFold를 자주 사용함.
- `cross_val_score()`는 하나의 지표를 간단히 확인할 때 사용함.
- `cross_validate()`는 여러 지표와 시간 정보를 함께 확인할 때 사용함.
- 테스트 데이터는 모델 선택과 튜닝에 반복 사용하지 않고 마지막 확인용으로 남겨야 함.